In [5]:
# Copyright (c) 2025 Marco Fago
#
# This code is licensed under the MIT License.
# See the LICENSE file in the repository for the full license text.

import uuid
from typing import Dict, Any, Optional

from google.adk.agents import Agent
from google.adk.runners import InMemoryRunner
from google.adk.tools import FunctionTool
from google.genai import types
from google.adk.events import Event

In [11]:
# --- Define Tool Functions ---
# These functions simulate the actions of the specialist agents.

def booking_handler(request: str) -> str:
    print("-------------------------- Booking Handler Called ----------------------------")
    return f"Booking action for '{request}' has been simulated."

def info_handler(request: str) -> str:
    print("-------------------------- Info Handler Called ----------------------------")
    return f"Information request for '{request}'. Result: Simulated information retrieval."

def unclear_handler(request: str) -> str:
    return f"Coordinator could not delegate request: '{request}'. Please clarify."

In [7]:
# --- Create Tools from Functions ---
booking_tool = FunctionTool(booking_handler)
info_tool = FunctionTool(info_handler)

# Define specialized sub-agents equipped with their respective tools
booking_agent = Agent(
    name="Booker",
    model="gemini-2.5-flash-lite",
    description="A specialized agent that handles all flight and hotel booking requests by calling the booking tool.",
    tools=[booking_tool]
)

info_agent = Agent(
    name="Info",
    model="gemini-2.5-flash-lite",
    description="A specialized agent that provides general information and answers user questions by calling the info tool.",
    tools=[info_tool]
)

# Define the parent agent with explicit delegation instructions
coordinator = Agent(
    name="Coordinator",
    model="gemini-2.5-flash-lite",
    instruction=(
        "You are the main coordinator. Your only task is to analyze incoming user requests "
        "and delegate them to the appropriate specialist agent. Do not try to answer the user directly.\n"
        "- For any requests related to booking flights or hotels, delegate to the 'Booker' agent.\n"
        "- For all other general information questions, delegate to the 'Info' agent."
    ),
    description="A coordinator that routes user requests to the correct specialist agent.",
    # The presence of sub_agents enables LLM-driven delegation (Auto-Flow) by default.
    sub_agents=[booking_agent, info_agent]
)

In [12]:
# --- Execution Logic ---

async def run_coordinator(runner: Runner, request: str):
    print(f"\n--- Running Coordinator with request: '{request}' ---")
    final_result = ""
    try:
        user_id = "user_123"
        session_id = str(uuid.uuid4()) # Fresh session for every request
        
        # 2. MUST AWAIT the session creation
        await runner.session_service.create_session(
            app_name=runner.app_name, user_id=user_id, session_id=session_id
        )

        # 3. MUST USE async for and run_async
        async for event in runner.run_async(
            user_id=user_id,
            session_id=session_id,
            new_message=types.Content(
                role='user',
                parts=[types.Part(text=request)]
            ),
        ):
            if event.is_final_response() and event.content:
                if hasattr(event.content, 'text') and event.content.text:
                     final_result = event.content.text
                elif event.content.parts:
                    text_parts = [part.text for part in event.content.parts if part.text]
                    final_result = "".join(text_parts)
                break

        return final_result
    except Exception as e:
        print(f"An error occurred while processing your request: {e}")
        return f"Error: {e}"

In [13]:
async def main():
    print("--- Google ADK Routing Example (Jupyter Async Style) ---")
    
    # Use standard Runner with InMemorySessionService
    session_service = InMemorySessionService()
    runner = Runner(
        agent=coordinator, 
        app_name="routing_app", 
        session_service=session_service
    )
    
    # 5. Await the execution of your queries
    result_a = await run_coordinator(runner, "Book me a hotel in Paris.")
    print(f"Final Output A: {result_a}")
    
    result_b = await run_coordinator(runner, "What is the highest mountain in the world?")
    print(f"Final Output B: {result_b}")
    
    result_c = await run_coordinator(runner, "Tell me a random fact.") 
    print(f"Final Output C: {result_c}")
    
    result_d = await run_coordinator(runner, "Find flights to Tokyo next month.")
    print(f"Final Output D: {result_d}")

In [14]:
await main()

Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.


--- Google ADK Routing Example (Jupyter Async Style) ---

--- Running Coordinator with request: 'Book me a hotel in Paris.' ---


Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.
Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.


-------------------------- Booking Handler Called ----------------------------


Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.


...

-------------------------- Info Handler Called ----------------------------


Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.


Final Output C: A group of flamingos is called a flamboyance.

--- Running Coordinator with request: 'Find flights to Tokyo next month.' ---


Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.
Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.


-------------------------- Booking Handler Called ----------------------------
Final Output D: Booking action for 'Find flights to Tokyo next month.' has been simulated.


## Test

In [1]:
import os
import asyncio
from google.genai import types
from google.adk.agents import Agent
from google.adk.runners import Runner
from google.adk.sessions import InMemorySessionService

os.environ["GOOGLE_GENAI_USE_VERTEXAI"] = "False"

In [4]:
print("--- Starting Jupyter ADK Test ---")

# 2. BARE MINIMUM AGENT
test_agent = Agent(
    name="Tester",
    model="gemini-2.5-flash-lite",
    instruction="Reply with exactly one word: SUCCESS"
)

# 3. CREATE THE SESSION SERVICE
session_service = InMemorySessionService()

# 4. AWAIT SESSION CREATION (Top-level await works in Jupyter!)
await session_service.create_session(
    app_name="test_app", 
    user_id="u1", 
    session_id="s1"
)

# 5. INITIALIZE RUNNER
runner = Runner(
    agent=test_agent,
    app_name="test_app",
    session_service=session_service
)

msg = types.Content(role="user", parts=[types.Part(text="Are you online?")])

print("Sending request to ADK...")

# 6. RUN IT
async for event in runner.run_async(user_id="u1", session_id="s1", new_message=msg):
    if event.is_final_response():
        # Safely grab the text
        response_text = "".join([p.text for p in event.content.parts if p.text])
        print(f"\nAGENT SAYS: {response_text}")

Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.


--- Starting Jupyter ADK Test ---
Sending request to ADK...

AGENT SAYS: SUCCESS
